In [1]:
!pip install pandas np

import pandas as pd
import numpy as np


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/b9/c5/fc1b368f303087d20e8c9bf3d6ceb186263cfac0ade735cd938538bea839/pandas-3.0.3-cp312-cp312-win_amd64.whl.metadata
  Using cached pandas-3.0.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Obtaining dependency information for numpy>=1.26.0 from https://files.pythonhosted.org/packages/ab/ca/feab00bd44aa5fe1ad2c18f08b4d3bb92e26484b0b1d1443897809ed528c/numpy-2.4.6-cp312-cp312-win_amd64.whl.metadata
  Using cached numpy-2.4.6-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Obtaining dependency information for tzdata from https://files.pythonhosted.org/packages/ce/e4/dccd7f47c4b64

In [2]:
data = [[1, 'Downtown Tech', 'New York'], [2, 'Suburb Mall', 'Chicago'], [3, 'City Center', 'Los Angeles'], [4, 'Corner Shop', 'Miami'], [5, 'Plaza Store', 'Seattle']]
stores = pd.DataFrame(data, columns=['store_id', 'store_name', 'location']).astype({'store_id': 'int64', 'store_name': 'string', 'location': 'string'})

stores = pd.concat([stores, pd.DataFrame(data,columns = stores.columns)])

data = [[1, 1, 'Laptop', 5, 999.99], [2, 1, 'Mouse', 50, 19.99], [3, 1, 'Keyboard', 25, 79.99], [4, 1, 'Monitor', 15, 299.99], [5, 2, 'Phone', 3, 699.99], [6, 2, 'Charger', 100, 25.99], [7, 2, 'Case', 75, 15.99], [8, 2, 'Headphones', 20, 149.99], [9, 3, 'Tablet', 2, 499.99], [10, 3, 'Stylus', 80, 29.99], [11, 3, 'Cover', 60, 39.99], [12, 4, 'Watch', 10, 299.99], [13, 4, 'Band', 25, 49.99], [14, 5, 'Camera', 8, 599.99], [15, 5, 'Lens', 12, 199.99]]
inventory = pd.DataFrame(data, columns=['inventory_id', 'store_id', 'product_name', 'quantity', 'price']).astype({'inventory_id': 'int64', 'store_id': 'int64', 'product_name': 'string', 'quantity': 'int64', 'price': 'float64'})

inventory = pd.concat([inventory, pd.DataFrame(data, columns = inventory.columns)])



In [4]:
stores

,store_id,store_name,location
0,1,Downtown Tech,New York
1,2,Suburb Mall,Chicago
2,3,City Center,Los Angeles
3,4,Corner Shop,Miami
4,5,Plaza Store,Seattle
0,1,Downtown Tech,New York
1,2,Suburb Mall,Chicago
2,3,City Center,Los Angeles
3,4,Corner Shop,Miami
4,5,Plaza Store,Seattle


In [7]:
merged_df = pd.merge(stores, inventory, on = 'store_id', how = 'left')
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   store_id      60 non-null     int64  
 1   store_name    60 non-null     string 
 2   location      60 non-null     string 
 3   inventory_id  60 non-null     int64  
 4   product_name  60 non-null     string 
 5   quantity      60 non-null     int64  
 6   price         60 non-null     float64
dtypes: float64(1), int64(3), string(3)
memory usage: 3.4 KB


In [31]:
merged_df = pd.merge(stores, inventory, on = 'store_id', how = 'left')

product_counts = merged_df.groupby(['store_id','store_name','location']).agg(
    no_of_products = ('product_name','count')
).reset_index()
product_counts

cheapest_product = merged_df.sort_values('price', ascending = True).\
    groupby(['store_id','store_name','location']).first().reset_index().\
    rename(
        columns = {	
            "product_name":"cheapest_product",	
            "quantity":"cheapest_product_quanity"
        }
    )[['store_id','store_name','location','cheapest_product','cheapest_product_quanity']]


most_expensive_product = merged_df.sort_values('price', ascending = False).\
    groupby(['store_id','store_name','location']).first().reset_index().\
    rename(
        columns = {	
            "product_name":"most_expensive_product",	
            "quantity":"most_expensive_product_quanity"
        }
    )[['store_id','store_name','location','most_expensive_product','most_expensive_product_quanity']]


final_df=pd.merge(product_counts,\
    pd.merge(cheapest_product,most_expensive_product, on=['store_id','store_name','location'], how='left'),\
    on=['store_id','store_name','location'],how='left')

final_df = final_df[(final_df['cheapest_product_quanity'] >=3) & (final_df['cheapest_product_quanity']> final_df['most_expensive_product_quanity'])]

final_df['imbalance_ratio'] = (final_df['cheapest_product_quanity'] / final_df['most_expensive_product_quanity']).round(2)

final_df[['store_id','store_name','location','most_expensive_product','cheapest_product','imbalance_ratio']].\
    sort_values(['imbalance_ratio','store_id'],ascending=[False,True])



,store_id,store_name,location,most_expensive_product,cheapest_product,imbalance_ratio
2,3,City Center,Los Angeles,Tablet,Stylus,40.0
1,2,Suburb Mall,Chicago,Phone,Case,25.0
0,1,Downtown Tech,New York,Laptop,Mouse,10.0
3,4,Corner Shop,Miami,Watch,Band,2.5
4,5,Plaza Store,Seattle,Camera,Lens,1.5


In [33]:

merged_df = stores.merge(inventory, on='store_id', how='left')

group_cols = ['store_id', 'store_name', 'location']

# Product counts
product_counts = (
    merged_df.groupby(group_cols)
    .size()
    .reset_index(name='no_of_products')
)

# Indices of cheapest and most expensive products
cheapest_idx = merged_df.groupby(group_cols)['price'].idxmin()
expensive_idx = merged_df.groupby(group_cols)['price'].idxmax()

cheapest_product = (
    merged_df.loc[cheapest_idx,
        group_cols + ['product_name', 'quantity']]
    .rename(columns={
        'product_name': 'cheapest_product',
        'quantity': 'cheapest_product_quantity'
    })
)

most_exp_product = (
    merged_df.loc[expensive_idx,
        group_cols + ['product_name', 'quantity']]
    .rename(columns={
        'product_name': 'most_exp_product',
        'quantity': 'most_exp_product_quantity'
    })
)

final_df = (
    product_counts
    .merge(cheapest_product, on=group_cols)
    .merge(most_exp_product, on=group_cols)
)

final_df = final_df[
    (final_df['no_of_products'] >= 3)
    &
    (
        final_df['cheapest_product_quantity']
        >
        final_df['most_exp_product_quantity']
    )
]

final_df['imbalance_ratio'] = (
    final_df['cheapest_product_quantity']
    / final_df['most_exp_product_quantity']
).round(2)

(
    final_df[
        [
            'store_id',
            'store_name',
            'location',
            'most_exp_product',
            'cheapest_product',
            'imbalance_ratio'
        ]
    ]
    .sort_values(
        ['imbalance_ratio', 'store_name'],
        ascending=[False, True]
    )
)

,store_id,store_name,location,most_exp_product,cheapest_product,imbalance_ratio
2,3,City Center,Los Angeles,Tablet,Stylus,40.0
1,2,Suburb Mall,Chicago,Phone,Case,25.0
0,1,Downtown Tech,New York,Laptop,Mouse,10.0
3,4,Corner Shop,Miami,Watch,Band,2.5
4,5,Plaza Store,Seattle,Camera,Lens,1.5
